![Kayak](https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png)

# Plan your trip with Kayak 

## Company's description 📇

<a href="https://www.kayak.com" target="_blank">Kayak</a> is a travel search engine that helps user plan their next trip at the best price.

The company was founded in 2004 by Steve Hafner & Paul M. English. After a few rounds of fundraising, Kayak was acquired by <a href="https://www.bookingholdings.com/" target="_blank">Booking Holdings</a> which now holds: 

* <a href="https://booking.com/" target="_blank">Booking.com</a>
* <a href="https://kayak.com/" target="_blank">Kayak</a>
* <a href="https://www.priceline.com/" target="_blank">Priceline</a>
* <a href="https://www.agoda.com/" target="_blank">Agoda</a>
* <a href="https://Rentalcars.com/" target="_blank">RentalCars</a>
* <a href="https://www.opentable.com/" target="_blank">OpenTable</a>

With over \$300 million revenue a year, Kayak operates in almost all countries and all languages to help their users book travels accros the globe. 

## Project 🚧

The marketing team needs help on a new project. After doing some user research, the team discovered that **70% of their users who are planning a trip would like to have more information about the destination they are going to**. 

In addition, user research shows that **people tend to be defiant about the information they are reading if they don't know the brand** which produced the content. 

Therefore, Kayak Marketing Team would like to create an application that will recommend where people should plan their next holidays. The application should be based on real data about:

* Weather 
* Hotels in the area 

The application should then be able to recommend the best destinations and hotels based on the above variables at any given time. 

## Goals 🎯

As the project has just started, your team doesn't have any data that can be used to create this application. Therefore, your job will be to: 

* Scrape data from destinations 
* Get weather data from each destination 
* Get hotels' info about each destination
* Store all the information above in a data lake
* Extract, transform and load cleaned data from your datalake to a data warehouse

## Scope of this project 🖼️

Marketing team wants to focus first on the best cities to travel to in France. According <a href="https://one-week-in.com/35-cities-to-visit-in-france/" target="_blank">One Week In.com</a> here are the top-35 cities to visit in France: 

```python 
["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]
```

Your team should focus **only on the above cities for your project**. 


## Helpers 🦮

To help you achieve this project, here are a few tips that should help you

### Get weather data with an API 

*   Use https://nominatim.org/ to get the gps coordinates of all the cities (no subscription required) Documentation : https://nominatim.org/release-docs/develop/api/Search/

*   Use https://openweathermap.org/appid (you have to subscribe to get a free apikey) and https://openweathermap.org/api/one-call-api to get some information about the weather for the 35 cities and put it in a DataFrame

*   Determine the list of cities where the weather will be the nicest within the next 7 days For example, you can use the values of daily.pop and daily.rain to compute the expected volume of rain within the next 7 days... But it's only an example, actually you can have different opinions on a what a nice weather would be like 😎 Maybe the most important criterion for you is the temperature or humidity, so feel free to change the rules !

*   Save all the results in a `.csv` file, you will use it later 😉 You can save all the informations that seem important to you ! Don't forget to save the name of the cities, and also to create a column containing a unique identifier (id) of each city (this is important for what's next in the project)

*   Use plotly to display the best destinations on a map

### Scrape Booking.com 

Since BookingHoldings doesn't have aggregated databases, it will be much faster to scrape data directly from booking.com 

You can scrap as many information asyou want, but we suggest that you get at least:

*   hotel name,
*   Url to its booking.com page,
*   Its coordinates: latitude and longitude
*   Score given by the website users
*   Text description of the hotel


### Create your data lake using S3 

Once you managed to build your dataset, you should store into S3 as a csv file. 

### ETL 

Once you uploaded your data onto S3, it will be better for the next data analysis team to extract clean data directly from a Data Warehouse. Therefore, create a SQL Database using AWS RDS, extract your data from S3 and store it in your newly created DB. 

## Deliverable 📬

To complete this project, your team should deliver:

* A `.csv` file in an S3 bucket containing enriched information about weather and hotels for each french city

* A SQL Database where we should be able to get the same cleaned data from S3 

* Two maps where you should have a Top-5 destinations and a Top-20 hotels in the area. You can use plotly or any other library to do so. It should look something like this: 

![Map](https://full-stack-assets.s3.eu-west-3.amazonaws.com/images/Kayak_best_destination_project.png)

# 1. Get weather data with an API 
## 1.1 : get the coordinates of the cities :

*   Use https://nominatim.org/ to get the gps coordinates of all the cities (no subscription required) Documentation : https://nominatim.org/release-docs/develop/api/Search/

*   Use https://openweathermap.org/appid (you have to subscribe to get a free apikey) and https://openweathermap.org/api/one-call-api to get some information about the weather for the 35 cities and put it in a DataFrame

*   Determine the list of cities where the weather will be the nicest within the next 7 days For example, you can use the values of daily.pop and daily.rain to compute the expected volume of rain within the next 7 days... But it's only an example, actually you can have different opinions on a what a nice weather would be like 😎 Maybe the most important criterion for you is the temperature or humidity, so feel free to change the rules !

*   Save all the results in a `.csv` file, you will use it later 😉 You can save all the informations that seem important to you ! Don't forget to save the name of the cities, and also to create a column containing a unique identifier (id) of each city (this is important for what's next in the project)

*   Use plotly to display the best destinations on a map


In [ ]:
import json
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import warnings
import logging  # for logging debug messages

# activate debug messages 
logger = logging.getLogger(__name__)

config_cities_list = ["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

# parameters setting 

load_dotenv()
WEATHER_API_KEY = os.getenv("API_KEY")

BASE_URL_NOMINATIM = "https://nominatim.openstreetmap.org/search"
BASE_URL_OPENWEATHERMAP="https://api.openweathermap.org/data/2.5"

CONFIG_CITIES_FILE = "cities.csv"
MAX_CITIES_NUMBER = 35

DATA_DIR = "data"
DATA_DIR_CONFIG = f"{DATA_DIR}/config"
os.makedirs(DATA_DIR_CONFIG, exist_ok=True)

DATA_DIR_CSV = f"{DATA_DIR}/csv"
os.makedirs(DATA_DIR_CSV, exist_ok=True)

DATA_DIR_HTML = f"{DATA_DIR}/html"
os.makedirs(DATA_DIR_HTML, exist_ok=True)

DATA_DIR_JSON = f"{DATA_DIR}/json"
os.makedirs(DATA_DIR_JSON, exist_ok=True)

DATA_DIR_JSON_CITIES = f"{DATA_DIR_JSON}/cities"
os.makedirs(DATA_DIR_JSON_CITIES, exist_ok=True)

DATA_DIR_JSON_HOTELS = f"{DATA_DIR_JSON}/hotels"
os.makedirs(DATA_DIR_JSON_HOTELS, exist_ok=True)

VALID_TYPES = {"city", "town", "village", "hamlet", "municipality", "islet", "historic", "tourism", "county", "gorge"}

USER_AGENT = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"

HEADERS = {'User-Agent': USER_AGENT}
DEFAULT_PARAMS = {"format": "json"}


# save first cities list to .csv (uncomment to save your first file !): 
# cities_list_df = pd.DataFrame({'city_name':config_cities_list})
# cities_list_df.to_csv (f"{DATA_DIR_CONFIG}/cities.csv",index=False, header=False)




In [2]:
# #cities_list_df = pd.DataFrame(cities_list)
# cities_list_df = pd.DataFrame({'city_name':config_cities_list})
# cities_list_df.to_csv (f"{DATA_DIR_CONFIG}/cities.csv",index=False, header=False)


In [3]:
# cities_list_df2 = pd.read_csv(f"{DATA_DIR_CONFIG}/cities.csv",header=None, names = ['city_name'])
# display(cities_list_df2)
# #


In [4]:
#weather_api_key

NameError: name 'weather_api_key' is not defined

In [5]:

# search gps coordinates for the 35 cities : 

import json
import random
import time

from django.conf.locale import de

def load_cities()-> pd.DataFrame :
    """load cities from a file or directly inside the function
    with column ['city_name']
    35 bst cities to visit in France
    """
    
    cities_list = pd.read_csv(f"{DATA_DIR_CONFIG}/{CONFIG_CITIES_FILE}",
                              header=None, names = ['city_name'])
    cities_list['city_name_plus'] = (
    cities_list["city_name"]
    .str.replace("'", "+", regex=False)
    .str.replace(" ", "+", regex=False)
    .str.replace("_", "+", regex=False)
    )
    
    cities_list['city_name_'] = (
        cities_list["city_name_plus"]
        .str.replace("+", "_", regex=False)
    )
    
    # add warning if more than 35 cities
    if len(cities_list)>MAX_CITIES_NUMBER:
        msg = f"More than {MAX_CITIES_NUMBER} in cities file from {DATA_DIR_CONFIG}/{CONFIG_CITIES_FILE}"
        warnings.warn(msg, UserWarning, stacklevel=2)
    return cities_list

def clean_name(name):
    """ clean name of city for name of file saving and city_id string
    replace "+" "'" & space by "_" 
    """
    name = name.replace(" ","_")
    name = name.replace("'",'_')
    name = name.replace('+','_')
    return name
    

def plus_name(name):
    """ plus name of city for search & query activities
    replace "_" "'" & space by "+" 
    """
    name = name.replace("'",'+')
    name = name.replace(" ","+")
    name = name.replace('_','+')
    return name

def _get_city_properties(data: list) -> dict | None:
    """get proper item in data from big json delivered by nominatim
    to extract later lat + lon properties  """
    for item in data:
        if item.get("addresstype") in VALID_TYPES:
            return item
    return None

def save_to_cache(city_name_: str, data):
    filename = f"{city_name_}.json"
    p = os.path.join(DATA_DIR_JSON_CITIES, filename)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)



def _cache_lookup(city_name_: str):
    """Return path of existing cache file for city_name_, or None."""
    p = os.path.join(DATA_DIR_JSON_CITIES, f"{city_name_}.json")
    return p if os.path.exists(p) else None


def load_from_cache(city_name_: str):
    """Load city data from json file if exists in cache dir."""
    
    data = _cache_lookup(city_name_)
    if data:
        with open(data, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except Exception:
                return None
    return None

def clean_cities_names(cities_list):
    cleaned_cities_list = []
    for city in cities_list:
        city_enc =city.replace(' ','+')
        cleaned_cities_list.append(city_enc)
    return cleaned_cities_list

def extract_coordinates(data, feature_index=0):
    """Extrait les coordonnées [lon, lat] d'une feature GeoJSON."""
    try:
        coords = data["features"][feature_index]["geometry"]["coordinates"]
        lon, lat = coords[0], coords[1]
        return lon, lat
    except (KeyError, IndexError, TypeError) as e:
        raise ValueError(f"Impossible d'extraire les coordonnées: {e}")

# # Depuis un fichier
# with open("data.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# lon, lat = extract_coordinates(data, feature_index=0)
# print(f"Longitude: {lon}, Latitude: {lat}")

# Bonus : extraire toutes les features avec leur nom
def extract_all_coordinates(data):
    """Retourne une liste de (nom, lon, lat) pour toutes les features."""
    results = []
    for feat in data.get("features", []):
        try:
            name = feat["properties"].get("name", "N/A")
            lon, lat = feat["geometry"]["coordinates"]
            results.append((name, lon, lat))
        except (KeyError, TypeError):
            continue
    return results

def get_nominatim(city_cleaned:str, delay:int = 1.82, max_retries: int = 3, timeout: int = 10):
    """ get city coordinates from nominatim api 
    city_cleaned : str : spaces replaced by '+' 
    delay : delay from request order , default is 1.82 s 
    """

    params = DEFAULT_PARAMS.copy()
    params['q'] = city_cleaned
    params['countrycodes'] = 'fr'
    for attempt in range(max_retries):
        try:
            
            response = requests.get(
                url = BASE_URL_NOMINATIM, 
                headers = HEADERS,
                params = params, 
                timeout=10            
                )
        except (requests.exceptions.Timeout, 
                requests.exceptions.CorrectionError) as e:
            if attempt < max_retries - 1:
                time.sleep(wait)
                continue
            print(f"[get_nominatim] échec après {max_retries} tentatives ({city_cleaned!r}) : {e}")
            return None
        status = response.status_code
    
        if status == 200:
            try:
                data = response.json()
            except ValueError:
                data = None
            time.sleep(delay)
            
            return data or None
            
        if status == 429 or status >= 500:
            wait = delay *2 + random.uniform(0,0.52) 
            if attempt < max_retries -1:
                time.sleep(wait)
                continue
            return None
    
        # others codes (400, 403) : no retry , error is definitive
        return None
    return None
            


In [6]:
    
# for name, lon, lat in extract_all_coordinates(data):
#     print(f"{name}: ({lon}, {lat})")

# ==============================================================
# *** main *** 
# ==============================================================
#gps_url = "https://nominatim.openstreetmap.org/search"

cities_list_df = load_cities()
cities_list_df


,city_name,city_name_plus,city_name_
0,Mont Saint Michel,Mont+Saint+Michel,Mont_Saint_Michel
1,St Malo,St+Malo,St_Malo
2,Bayeux,Bayeux,Bayeux
3,Le Havre,Le+Havre,Le_Havre
4,Rouen,Rouen,Rouen
5,Paris,Paris,Paris
6,Amiens,Amiens,Amiens
7,Lille,Lille,Lille
8,Strasbourg,Strasbourg,Strasbourg
9,Chateau du Haut Koenigsbourg,Chateau+du+Haut+Koenigsbourg,Chateau_du_Haut_Koenigsbourg


In [8]:

# cleaned_cities = clean_cities_names(cities_list)
# print(cleaned_cities)
def get_coordinates_cities(cities_list_df) -> pd.DataFrame:
    """ get coordinates for all cities from dataframe to dataframe """
    results = []
    for idx,row in cities_list_df.iterrows():
        city = row['city_name']
        city_ = row['city_name_']
        city_plus = row['city_name_plus']
        
        # city = row['city_name']
        
        # city_plus = row['city_name_plus']
        # city_ = row['city_name_']
        
        #https://nominatim.openstreetmap.org/search?<params>
        #city_enc =city.replace(' ','+')
        # params = "?q="+city
        # format = "&format=geojson"
        # input_url = gps_url+params+format
        # print(input_url)
        #response = requests.get(input_url)
        
        # read from cache if city is already there 
        # if yes : take data from cache in same json format as direct call from nominatim: 
        data = load_from_cache(city_)
        if data is None:
        # else : get coordinates:
            data = get_nominatim(city_plus)
            
        if data:
            save_to_cache(city_,data)
            # print for debug
            logger.debug("%s coord : %s", city, data)   # <-- remplace print()
    
            # save data in cache :
            item = _get_city_properties(data)
            if item:
                results.append({
                    "city_name"  : city,
                    "city_name_" : city_,
                    "city_plus"  : city_plus,
                    "lat": item.get("lat"),
                    "lon": item.get("lon"),
                    
                })
            else:
                # item is None : 
                results.append({
                                    "city_name"  : city,
                                    "city_name_" : city_,
                                    "city_plus"  : city_plus,
                                    "lat": None,
                                    "lon": None
                            })
        else:
            # data is none : report error in reading coordinates 
            results.append({
                    "city_name"  : city,
                    "city_name_" : city_,
                    "city_plus"  : city_plus,
                    "lat": None,
                    "lon": None
            })
               
    results_df = pd.DataFrame(results)
    return results_df
   
        
    #print(response.json())      

results_df = get_coordinates_cities(cities_list_df)  
results_df      

Mont Saint Michel coord : [{'place_id': 285390554, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 211285890, 'lat': '48.6359541', 'lon': '-1.5114600', 'class': 'place', 'type': 'islet', 'place_rank': 20, 'importance': 0.4711675122514022, 'addresstype': 'islet', 'name': 'Mont Saint-Michel', 'display_name': 'Mont Saint-Michel, Le Mont-Saint-Michel, Avranches, Manche, Normandie, France métropolitaine, 50170, France', 'boundingbox': ['48.6349172', '48.6370310', '-1.5133292', '-1.5094796']}, {'place_id': 120517360, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 11981568929, 'lat': '48.6906680', 'lon': '5.8818463', 'class': 'natural', 'type': 'peak', 'place_rank': 18, 'importance': 0.16006674219882505, 'addresstype': 'peak', 'name': 'Mont Saint-Michel', 'display_name': 'Mont Saint-Michel, Toul, Meurthe-et-Moselle, Grand Est, France métropolitaine, 54200, France

,city_name,city_name_,city_plus,lat,lon
0,Mont Saint Michel,Mont_Saint_Michel,Mont+Saint+Michel,48.6359541,-1.5114600
1,St Malo,St_Malo,St+Malo,48.6495180,-2.0260409
2,Bayeux,Bayeux,Bayeux,49.2764624,-0.7024738
3,Le Havre,Le_Havre,Le+Havre,49.4938975,0.1079732
4,Rouen,Rouen,Rouen,49.4404591,1.0939658
5,Paris,Paris,Paris,48.8534951,2.3483915
6,Amiens,Amiens,Amiens,49.8941708,2.2956951
7,Lille,Lille,Lille,50.6365654,3.0635282
8,Strasbourg,Strasbourg,Strasbourg,48.5846140,7.7507127
9,Chateau du Haut Koenigsbourg,Chateau_du_Haut_Koenigsbourg,Chateau+du+Haut+Koenigsbourg,48.2493820,7.3439412


In [ ]:
results

[{'city_name': 'Mont+Saint+Michel', 'lat': '48.6359541', 'lon': '-1.5114600'},
 {'city_name': 'St+Malo', 'lat': '48.6495180', 'lon': '-2.0260409'},
 {'city_name': 'Bayeux', 'lat': '49.2764624', 'lon': '-0.7024738'},
 {'city_name': 'Le+Havre', 'lat': '49.4938975', 'lon': '0.1079732'},
 {'city_name': 'Rouen', 'lat': '49.4404591', 'lon': '1.0939658'},
 {'city_name': 'Paris', 'lat': '48.8534951', 'lon': '2.3483915'},
 {'city_name': 'Amiens', 'lat': '49.8941708', 'lon': '2.2956951'},
 {'city_name': 'Lille', 'lat': '50.6365654', 'lon': '3.0635282'},
 {'city_name': 'Strasbourg', 'lat': '48.5846140', 'lon': '7.7507127'},
 {'city_name': 'Chateau+du+Haut+Koenigsbourg',
  'lat': '48.2493820',
  'lon': '7.3439412'},
 {'city_name': 'Colmar', 'lat': '48.0777517', 'lon': '7.3579641'},
 {'city_name': 'Eguisheim', 'lat': '48.0447968', 'lon': '7.3079618'},
 {'city_name': 'Besancon', 'lat': '47.2380222', 'lon': '6.0243622'},
 {'city_name': 'Dijon', 'lat': '47.3215806', 'lon': '5.0414701'},
 {'city_name':

In [10]:

#DATA_DIR_CSV = ./json
results_df.to_csv(os.path.join(DATA_DIR_CSV, "cities.csv"), index=False, encoding="utf-8")
print("Saved", len(results_df), "rows to", os.path.join(DATA_DIR_CSV, "cities.csv"))

Saved 35 rows to data/csv\cities.csv


## 1.2 : weather forecasts
*   Use https://openweathermap.org/appid (you have to subscribe to get a free apikey) and https://openweathermap.org/api/one-call-api to get some information about the weather for the 35 cities and put it in a DataFrame

*   Determine the list of cities where the weather will be the nicest within the next 7 days For example, you can use the values of daily.pop and daily.rain to compute the expected volume of rain within the next 7 days... But it's only an example, actually you can have different opinions on a what a nice weather would be like 😎 Maybe the most important criterion for you is the temperature or humidity, so feel free to change the rules !

*   Save all the results in a `.csv` file, you will use it later 😉 You can save all the informations that seem important to you ! Don't forget to save the name of the cities, and also to create a column containing a unique identifier (id) of each city (this is important for what's next in the project)

*   Use plotly to display the best destinations on a map


In [13]:
results1=results_df.loc[1]

In [ ]:
# get weather forecasts of all cities by calling the api 

# determine the top 7 cities with the best weather conditions for next 7 days
#  daily.pop and daily.rain : expected volume of rain in next days 
# temperature + humidity 

# API call 2.5 soon deprecated:
#https://api.openweathermap.org/data/2.5/onecall?lat={lat}&lon={lon}&exclude={part}&appid={API key}

# API call 4.0

# https://api.openweathermap.org/data/4.0/onecall/current?lat={lat}&lon={lon}&appid={API key}

#WEATHER_API_KEY = weather_api_key


# adds for tests
lat = results1['lat']
lon = results1['lon']
#https://api.openweathermap.org/data/2.5/onecall/current?lat=48.6495180&lon=-2.0260409&units=metric&lang=en&appid={'e6c25cd16e57edea5d19d272bf7c6a91'}



def get_weather(lat:float, lon:float) -> dict :
    """ get weatherforecast from openweathermap
    """
    params = {
        "lat": lat, "lon": lon,
        #"cnt": 100,
        "units": "metric",
        "appid": WEATHER_API_KEY,
    }
    response = requests.get(url=f"{BASE_URL_OPENWEATHERMAP}/forecast" ,
                            params = params,
                            timeout = 10)
    if response.status_code == 200:
        return response.json()
    print(f"  openweathermap error {response.status_code}: {response.text[:200]}")
    return None

def compile_weather(weather_data)-> pd.DataFrame:
    """ compile in a dataFrame weather data 
    coming from call weather_api
    that is in json format"""

    df_weather = pd.json_normalize(
        weather_data["list"],
        sep="_",
        record_path=["weather"],      # extrait weather (liste imbriquée) comme colonnes séparées
        meta=[
            "dt", "dt_txt", "visibility", "pop",
            ["main", "temp"], ["main", "feels_like"], ["main", "temp_min"], ["main", "temp_max"],
            ["main", "pressure"], ["main", "humidity"], ["main", "dew_point"],
            ["wind", "speed"], ["wind", "deg"], ["wind", "gust"],
            ["clouds", "all"], ["sys", "pod"],
        ],
        meta_prefix="",
        errors="ignore",              # tolère les champs absents (ex: pas de gust sur certaines entrées)
        record_prefix="weather_",
    )

    df_weather["datetime"] = pd.to_datetime(df_weather["dt_txt"])
    df_weather = df_weather.sort_values("datetime").reset_index(drop=True)
    return df_weather



In [15]:
pd.set_option("display.max_columns", None)   # affiche toutes les colonnes
pd.set_option("display.width", None)         # pas de troncature de largeur (auto-adapte au terminal/notebook)
pd.set_option("display.max_colwidth", None)  # pas de troncature du contenu des cellules
pd.set_option("display.max_rows", 100)       # optionnel : plus de lignes visibles (défaut 60)


In [24]:

def compile_weather2(data):
    df = pd.json_normalize(data["list"], sep="_")
    # df["weather_main"] = df["weather"].apply(lambda w: w[0]["main"] if w else None)
    # df["weather_description"] = df["weather"].apply(lambda w: w[0]["description"] if w else None)
    # df = df.drop(columns="weather")
    
    # 2eme iter : 
    # Extraction du champ weather (liste imbriquée avec 1 élément par entrée)
    weather_df = pd.json_normalize(df["weather"].apply(lambda w: w[0] if w else {}))
    weather_df = weather_df.add_prefix("weather_")

    df = pd.concat([df.drop(columns="weather"), weather_df], axis=1)
    df["datetime"] = pd.to_datetime(df["dt_txt"])
    df = df.sort_values("datetime").reset_index(drop=True)

        
    return df
   
   
weather_forecast = get_weather(lat,lon)    
print(weather_forecast)
df_weather2 = compile_weather2(weather_forecast)

display(df_weather2.head())

{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1790067600, 'main': {'temp': 18.29, 'feels_like': 18.05, 'temp_min': 18.26, 'temp_max': 18.29, 'pressure': 1030, 'sea_level': 1030, 'grnd_level': 1028, 'humidity': 72, 'temp_kf': 0.03, 'dew_point': 13.16}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'clouds': {'all': 2}, 'wind': {'speed': 1.54, 'deg': 100, 'gust': 3.76}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'd'}, 'dt_txt': '2026-09-22 09:00:00'}, {'dt': 1790078400, 'main': {'temp': 19.02, 'feels_like': 18.81, 'temp_min': 19.02, 'temp_max': 20.48, 'pressure': 1030, 'sea_level': 1030, 'grnd_level': 1027, 'humidity': 70, 'temp_kf': -1.46, 'dew_point': 13.43}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'clouds': {'all': 1}, 'wind': {'speed': 3.46, 'deg': 26, 'gust': 6.42}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'd'}, 'dt_txt': '2026-09-22 12:00:00'}, {'dt': 1790089200, 'main': {'temp':

,dt,visibility,pop,dt_txt,main_temp,main_feels_like,main_temp_min,main_temp_max,main_pressure,main_sea_level,main_grnd_level,main_humidity,main_temp_kf,main_dew_point,clouds_all,wind_speed,wind_deg,wind_gust,sys_pod,weather_id,weather_main,weather_description,weather_icon,datetime
0,1790067600,10000.0,0,2026-09-22 09:00:00,18.29,18.05,18.26,18.29,1030,1030,1028,72,0.03,13.16,2,1.54,100,3.76,d,800,Clear,clear sky,01d,2026-09-22 09:00:00
1,1790078400,10000.0,0,2026-09-22 12:00:00,19.02,18.81,19.02,20.48,1030,1030,1027,70,-1.46,13.43,1,3.46,26,6.42,d,800,Clear,clear sky,01d,2026-09-22 12:00:00
2,1790089200,10000.0,0,2026-09-22 15:00:00,20.42,20.19,20.42,21.48,1028,1028,1026,64,-1.06,13.39,1,4.16,18,7.09,d,800,Clear,clear sky,01d,2026-09-22 15:00:00
3,1790100000,10000.0,0,2026-09-22 18:00:00,19.04,18.83,19.04,19.04,1025,1025,1024,70,0.00,14.49,0,3.83,15,5.72,d,800,Clear,clear sky,01d,2026-09-22 18:00:00
4,1790110800,10000.0,0,2026-09-22 21:00:00,18.88,18.78,18.88,18.88,1026,1026,1024,75,0.00,14.64,0,3.66,67,5.44,n,800,Clear,clear sky,01n,2026-09-22 21:00:00


In [ ]:
# df_cities = pd.read_csv(os.path.join(DATA_DIR_CSV, "cities.csv"), encoding="utf-8")

# #weather_forecast = get_weather(lat, lon)

In [25]:
def summarize_weather(df: pd.DataFrame, hour_start: int = 8, hour_end: int = 20) -> pd.DataFrame:
    """ summarize weather on 5 days forecast
    compute daily rain and daily pop (probability of precipitation) between hour_start and hour_end
    """
    # define criteria 
    # compute summary on several criteria 
    # determine the top 7 cities with the best weather conditions for next 7 days
    #  daily.pop and daily.rain : expected volume of rain in next days 
    # temperature + humidity 
    
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["date"] = df["datetime"].dt.date
    df["hour"] = df["datetime"].dt.hour

    mask = (df["hour"] >= hour_start) & (df["hour"] <= hour_end)
    day_df = df.loc[mask].copy()

    if "rain_3h" not in day_df.columns:
        day_df["rain_3h"] = 0
    if "pop" not in day_df.columns:
        day_df["pop"] = 0
        
    day_df['is_clear'] = day_df['weather_main'].isin(['Clear']) 
    day_df['is_rain'] = day_df['weather_main'].isin(['Rain', 'Drizzle', 'Thunderstorm'])
    day_df['main_temp'] = day_df['main_temp'].fillna(0)
    day_df['main_humidity'] = day_df['main_humidity'].fillna(0)
    day_df["rain_3h"] = day_df["rain_3h"].fillna(0)
    day_df["pop"] = day_df["pop"].fillna(0)
    day_df["wind_speed"] = day_df["wind_speed"].fillna(0)

    summary = day_df.groupby("date").agg(
        rain_sum=("rain_3h", "sum"),
        pop_max=("pop", "max"),
        temp_mean=("main_temp", "mean"),
        humidity_mean=("main_humidity", "mean"),
        clear_slots_count=("is_clear", "sum"),
        rain_slots_count=("is_rain", "sum"),
        wind_speed_max=("wind_speed", "max"),
        n_slots=("hour", "count"),
    ).reset_index()

    summary["clear_sky_ratio"] = summary["clear_slots_count"] / summary["n_slots"]
    summary["rain_ratio"] = summary["rain_slots_count"] / summary["n_slots"]


    # Critère booléen "jour agréable"
    summary["is_pleasant_day"] = (
        (summary["clear_sky_ratio"] >= 0.5)
        & summary["temp_mean"].between(15, 27)
        & (summary["humidity_mean"] < 70)
        & (summary["rain_sum"] < 0.5)
        & (summary["rain_slots_count"] == 0)
        & (summary["pop_max"] < 0.3)
        & (summary["wind_speed_max"] < 11)  # m/s ≈ 40 km/h
    )

    # Bonus : score continu (0 à 1) plutôt que binaire, plus nuancé
    summary["comfort_score"] = (
        0.30 * summary["clear_sky_ratio"]
        + 0.25 * (1 - (summary["humidity_mean"] / 100)).clip(0, 1)
        + 0.20 * (1 - summary["pop_max"]).clip(0, 1)
        + 0.15 * (1 - (summary["temp_mean"] - 21).abs() / 15).clip(0, 1)  # optimum ~21°C
        + 0.10 * (1 - (summary["wind_speed_max"] / 15)).clip(0, 1)
    ).clip(0, 1).round(2)

    return summary



    return summary


# daily_summary = summarize_weather(df_weather2)
# print(daily_summary)
    

    
    

In [26]:


def get_weather_data_for_cities(df_cities: pd.DataFrame) -> pd.DataFrame:
    results_weather = []
    for _, row in df_cities.iterrows():
        name = row['city_name']
        lat, lon = row['lat'], row['lon']
        
        # print(f"{name}\n")
        weather_forecast = get_weather(lat, lon)
        df_weather = compile_weather2(weather_forecast)
        
        sw = summarize_weather(df_weather)
        sw['city'] = name
        results_weather.append(sw)
        
        # display(sw)  # Optionally display the summary for each city
    
    return pd.concat(results_weather, ignore_index=True)
       


In [ ]:
# select top 7 cities with best weather conditions for next 5 days : 
df_cities = results_df
result_weather = get_weather_data_for_cities(df_cities)
display(result_weather)


,date,rain_sum,pop_max,temp_mean,humidity_mean,clear_slots_count,rain_slots_count,wind_speed_max,n_slots,clear_sky_ratio,rain_ratio,is_pleasant_day,comfort_score,city
0,2026-09-22,0.0,0.0,19.9975,62.00,4,0,2.94,4,1.00,0.0,True,0.82,Mont Saint Michel
1,2026-09-23,0.0,0.0,19.7350,62.75,2,0,4.22,4,0.50,0.0,True,0.65,Mont Saint Michel
2,2026-09-24,0.0,0.0,20.9125,65.25,0,0,2.93,4,0.00,0.0,False,0.52,Mont Saint Michel
3,2026-09-25,0.0,0.0,20.4500,67.75,4,0,5.49,4,1.00,0.0,True,0.79,Mont Saint Michel
4,2026-09-26,0.0,0.0,19.3450,71.00,0,0,5.08,4,0.00,0.0,False,0.47,Mont Saint Michel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,2026-09-22,0.0,0.0,21.5775,40.75,1,0,6.08,4,0.25,0.0,False,0.63,La Rochelle
171,2026-09-23,0.0,0.0,24.6625,40.25,1,0,3.96,4,0.25,0.0,False,0.61,La Rochelle
172,2026-09-24,0.0,0.0,24.1150,51.00,0,0,3.65,4,0.00,0.0,False,0.52,La Rochelle
173,2026-09-25,0.0,0.0,21.8225,66.25,3,0,4.99,4,0.75,0.0,True,0.72,La Rochelle


In [29]:

date_span = result_weather["date"].unique()
date_span


array([datetime.date(2026, 9, 22), datetime.date(2026, 9, 23),
       datetime.date(2026, 9, 24), datetime.date(2026, 9, 25),
       datetime.date(2026, 9, 26)], dtype=object)

In [38]:
print(len(date_span))
date_span[1]

5


datetime.date(2026, 9, 23)

In [30]:


def select_best_weather_cities(df_weather_summary: pd.DataFrame, top_n: int = 7, start_date: pd.Timestamp = None, end_date: pd.Timestamp = None) -> pd.DataFrame:
    """Select the top N cities with the best weather conditions based on comfort_score.
    between optional dates """
    # Group by city and calculate the average comfort score over the forecast period
    mask = df_weather_summary['date'].between(pd.Timestamp.now().date(), pd.Timestamp.now().date() + pd.Timedelta(days=5))
    df_weather_sub = df_weather_summary.loc[mask].copy()
    city_scores = df_weather_sub.groupby("city").agg(
        avg_comfort_score=("comfort_score", "mean")
    ).reset_index()

    # Sort by average comfort score in descending order and select top N cities
    top_cities = city_scores.sort_values("avg_comfort_score", ascending=False)
    top_cities['selected'] = 0
    top_cities.loc[top_cities.index[:top_n], 'selected'] = 1

    return top_cities


top_cities = select_best_weather_cities(result_weather, top_n=7)
# merge left with df_cities to get lat/lon for top cities
top_cities = top_cities.merge(df_cities[['city_name', 'lat', 'lon']], left_on='city', right_on='city_name', how='left')
top_cities = top_cities.drop(columns=['city_name'])  # Drop the duplicate city_name column after merge
display(top_cities[:7])

,city,avg_comfort_score,selected,lat,lon
0,Lyon,0.768,1,45.7578137,4.8320114
1,Annecy,0.754,1,45.8992348,6.1288847
2,Colmar,0.742,1,48.0777517,7.3579641
3,Collioure,0.738,1,42.5250500,3.0831554
4,Carcassonne,0.738,1,43.2130358,2.3491069
5,Ariege,0.734,1,42.9455368,1.4065544
6,Strasbourg,0.722,1,48.5846140,7.7507127


*   Determine the list of cities where the weather will be the nicest within the next 7 days For example, you can use the values of daily.pop and daily.rain to compute the expected volume of rain within the next 7 days... But it's only an example, actually you can have different opinions on a what a nice weather would be like 😎 Maybe the most important criterion for you is the temperature or humidity, so feel free to change the rules !

*   Save all the results in a `.csv` file, you will use it later 😉 You can save all the informations that seem important to you ! Don't forget to save the name of the cities, and also to create a column containing a unique identifier (id) of each city (this is important for what's next in the project)

*   Use plotly to display the best destinations on a map


In [33]:
import plotly.express as px
# plot all cities wiith their average comfort score on a map of France
top_cities["marker_size"] = top_cities['selected'].map({1:15, 0:5})  # Example sizes for selected and non-selected cities
fig = px.scatter_mapbox(top_cities, 
                     lat="lat", 
                     lon="lon",
                     size="marker_size", 
                     size_max = 20,
                     hover_name="city", 
                     hover_data={"avg_comfort_score": ":.2f", "selected": True, "lat": False, "lon": False, "marker_size": False},
                     zoom=5,
                     center={"lat": 46.6, "lon": 2.5},
                     mapbox_style="open-street-map",
                     color="avg_comfort_score", 
                     color_continuous_scale=px.colors.sequential.Viridis, 
                     title="Top 7 Cities with Best Weather Conditions in France")
fig.update_layout(title="Indice de confort météo par ville", 
                  width = 900,
                  height=700,
                  margin=dict(l=0, r=0, t=40, b=0))
fig.show()


### Scrape Booking.com 

Since BookingHoldings doesn't have aggregated databases, it will be much faster to scrape data directly from booking.com 

You can scrap as many information asyou want, but we suggest that you get at least:

*   hotel name,
*   Url to its booking.com page,
*   Its coordinates: latitude and longitude
*   Score given by the website users
*   Text description of the hotel


In [ ]:
#https://www.booking.com/searchresults.fr.html?ss=Mont%2BSaint%2BMichel&ssne=Montauban&ssne_untouched=Montauban&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4ARfIAQ_YAQPoAQH4AQGIAgGoAgG4Arbzs9UGwAIB0gIkNzRkNzU5ZmUtZDM0MS00ZjIxLTlmZjctNjA1YjBiYjkzNWFi2AIB4AIB&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=900039327&dest_type=city&ac_meta=GhA2OGUxM2Q4YmYwMDcwYzY2IAAoATICZnI6EU1vbnQrU2FpbnQrTWljaGVs&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=68e13d8bf0070c66&checkin=2026-09-20&checkout=2026-09-21&group_adults=2&no_rooms=1&group_children=0

In [34]:
%load_ext autoreload
%autoreload 2  
# Recharge tous les modules avant chaque exécution de cellule

In [41]:
print(date_span[3])
print(date_span[4])



2026-09-25
2026-09-26


In [46]:
top_cities

,city,avg_comfort_score,selected,lat,lon,marker_size
0,Lyon,0.768,1,45.7578137,4.8320114,15
1,Annecy,0.754,1,45.8992348,6.1288847,15
2,Colmar,0.742,1,48.0777517,7.3579641,15
3,Collioure,0.738,1,42.5250500,3.0831554,15
4,Carcassonne,0.738,1,43.2130358,2.3491069,15
5,Ariege,0.734,1,42.9455368,1.4065544,15
6,Strasbourg,0.722,1,48.5846140,7.7507127,15
7,Montauban,0.722,0,44.0175835,1.3549991,5
8,Besancon,0.722,0,47.2380222,6.0243622,5
9,Chateau du Haut Koenigsbourg,0.714,0,48.2493820,7.3439412,5


In [47]:
import importlib
import booking_scraper_init
importlib.reload(booking_scraper_init)
from booking_scraper_init import get_hotel_availability, get_hotel_availability_sync


checkin_date = "2026-09-25"
checkout_date = "2026-09-26"
# default journey dates : tomorrow - end of datespan :
checkin_date = date_span[1].isoformat()   # "2026-09-22"
checkout_date = date_span[-1].isoformat() # "2026-09-26"
n_adults = 2
n_rooms = 1
max_results = 10


for city in top_cities.loc[top_cities['selected'] == 1, 'city']:
    print(f"Searching hotels in {city}...")
    results = get_hotel_availability_sync(
            city=city,
            checkin_date=checkin_date,
            checkout_date=checkout_date,
            group_adults=n_adults,
            no_rooms=n_rooms,
            max_results=max_results,
        )

    print(json.dumps(results, ensure_ascii=False, indent=2))

Searching hotels in Lyon...
[
  {
    "hotel_name": "Novotel Lyon Gerland",
    "url": "https://www.booking.com/hotel/fr/novotel-lyon-gerland.fr.html",
    "address": null,
    "hotel_description": "Situé à Lyon, sur les rives du Rhône, le Novotel Lyon Gerland Musée des Confluences vous accueille à 50 mètres de la salle de concert Halle Tony Garnier, à...",
    "city": null,
    "latitude": 45.733195,
    "longitude": 4.823688,
    "score": 7.6,
    "price": 106.0,
    "currency": "EUR",
    "room_description": "Chambre Double Classique avec Canapé\n2 lits (1 canapé-lit, 1 grand lit double)"
  },
  {
    "hotel_name": "Mama Shelter Lyon",
    "url": "https://www.booking.com/hotel/fr/mama-shelter-lyon.fr.html",
    "address": null,
    "hotel_description": "Situé à Lyon, le Mama Shelter Lyon propose une réception ouverte 24h/24, un restaurant et des suites.",
    "city": null,
    "latitude": 45.74695278493895,
    "longitude": 4.845267757773399,
    "score": 8.5,
    "price": 129.0,
  

In [ ]:
print(json.dumps(results, ensure_ascii=False, indent=2))

[
  {
    "name": "Novotel Lyon Gerland",
    "url": "https://www.booking.com/hotel/fr/novotel-lyon-gerland.fr.html",
    "latitude": 45.733195,
    "longitude": 4.823688,
    "score": 7.6,
    "description": "Chambre Classique avec 1 Lit Double et 1 Lit Simple\n2 lits (1 lit double, 1 canapé-lit)"
  },
  {
    "name": "Mama Shelter Lyon",
    "url": "https://www.booking.com/hotel/fr/mama-shelter-lyon.fr.html",
    "latitude": 45.74695278493895,
    "longitude": 4.845267757773399,
    "score": 8.5,
    "description": "Chambre Small Mama Double\n1 grand lit double\nPlus que 7 disponibles à ce tarif"
  },
  {
    "name": "Campanile PRIME Lyon Centre Gare Part-Dieu",
    "url": "https://www.booking.com/hotel/fr/campanile-lyon-centre-forum-part-dieu.fr.html",
    "latitude": 45.76027411252216,
    "longitude": 4.863327741622925,
    "score": 8.6,
    "description": "Chambre Double Standard\n1 lit double\nAnnulation gratuite"
  },
  {
    "name": "MEININGER Hotel Lyon Centre Berthelot",
   

In [ ]:
# function to aggregate hotels results : 

# function to select top 20 hotels in the cities selection :
# criteria 1 : top rated hotels 
# criteria2 :  